[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch05/NN_DL_ch05_CryptoPrediction/NN_DL_ch05_CryptoPrediction.ipynb)

# NN_DL_ch05_CryptoPrediction

**LSTM for Cryptocurrency Return & Crash Prediction**

Download real Bitcoin price data, compute returns, and train an LSTM for return forecasting and crash classification.

In [ ]:
!pip install -q torch matplotlib pandas scikit-learn scipy

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load Bitcoin Data
import pandas as pd

try:
    url = 'https://raw.githubusercontent.com/datasets/bitcoin/main/data/data.csv'
    btc = pd.read_csv(url)
    date_col = [c for c in btc.columns if 'date' in c.lower() or 'time' in c.lower()][0]
    price_col = [c for c in btc.columns if 'price' in c.lower() or 'close' in c.lower() or 'value' in c.lower()]
    if not price_col:
        price_col = [c for c in btc.columns if c != date_col]
    btc[date_col] = pd.to_datetime(btc[date_col])
    btc = btc.sort_values(date_col).dropna(subset=[price_col[0]])
    prices = btc[price_col[0]].values.astype(float)
    dates = btc[date_col].values
    print(f"Loaded real Bitcoin data: {len(prices)} observations")
except Exception as e:
    print(f"Could not load remote data ({e}), generating realistic crypto prices")
    np.random.seed(42)
    n = 2000
    returns = np.random.normal(0.001, 0.04, n)
    crash_idx = np.random.choice(n, 20, replace=False)
    returns[crash_idx] -= np.random.uniform(0.05, 0.15, 20)
    rally_idx = np.random.choice(n, 15, replace=False)
    returns[rally_idx] += np.random.uniform(0.05, 0.12, 15)
    prices = 10000 * np.exp(np.cumsum(returns))
    dates = pd.date_range('2018-01-01', periods=n, freq='D')
    print(f"Generated {n} synthetic crypto prices")

log_ret = np.diff(np.log(prices))
print(f"Returns: mean={log_ret.mean():.5f}, std={log_ret.std():.4f}, min={log_ret.min():.4f}, max={log_ret.max():.4f}")

In [ ]:
# Plot BTC Returns
from scipy.stats import norm

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

x_axis = dates[1:] if len(dates) == len(log_ret) + 1 else np.arange(len(log_ret))
axes[0].plot(x_axis, log_ret, color=MAIN_BLUE, lw=0.5)
axes[0].axhline(0, color='grey', lw=0.5)
axes[0].set_title('Bitcoin Log Returns', fontweight='bold')
axes[0].set_ylabel('Log Return')

crash_mask = log_ret < -0.05
crash_idx_plot = np.where(crash_mask)[0]
if len(crash_idx_plot) > 0:
    x_arr = np.array(x_axis)
    axes[0].scatter(x_arr[crash_idx_plot], log_ret[crash_idx_plot],
                    color=IDA_RED, s=20, zorder=5, label='Crashes (<-5%)')
    axes[0].legend()

axes[1].hist(log_ret, bins=80, color=MAIN_BLUE, edgecolor='white', density=True, alpha=0.7)
x_norm = np.linspace(log_ret.min(), log_ret.max(), 200)
axes[1].plot(x_norm, norm.pdf(x_norm, log_ret.mean(), log_ret.std()),
             color=IDA_RED, lw=2, label='Normal fit')
axes[1].set_title('Return Distribution (fat tails)', fontweight='bold')
axes[1].set_xlabel('Log Return'); axes[1].legend()

plt.tight_layout()
save_fig('nn_ch05_btc_returns')

In [ ]:
# Prepare Sequences for LSTM
from sklearn.preprocessing import StandardScaler

seq_len = 30
X_seq, y_seq = [], []
for i in range(seq_len, len(log_ret)):
    X_seq.append(log_ret[i-seq_len:i])
    y_seq.append(log_ret[i])

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32)

scaler = StandardScaler()
X_flat = scaler.fit_transform(X_seq)
X_seq_scaled = X_flat.reshape(-1, seq_len, 1)

split = int(0.8 * len(X_seq_scaled))
X_train = torch.FloatTensor(X_seq_scaled[:split])
X_test  = torch.FloatTensor(X_seq_scaled[split:])
y_train_reg = torch.FloatTensor(y_seq[:split])
y_test_reg  = y_seq[split:]

crash_threshold = -0.03
y_train_cls = torch.FloatTensor((y_seq[:split] < crash_threshold).astype(np.float32))
y_test_cls  = (y_seq[split:] < crash_threshold).astype(int)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Crash rate: train={y_train_cls.mean():.4f}, test={y_test_cls.mean():.4f}")

In [ ]:
# LSTM Model
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class CryptoLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=0.2)
        self.fc_return = nn.Linear(hidden_size, 1)
        self.fc_crash  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]
        return self.fc_return(last).squeeze(), self.fc_crash(last).squeeze()

model = CryptoLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
mse_loss = nn.MSELoss()
bce_loss = nn.BCEWithLogitsLoss()

train_ds = TensorDataset(X_train, y_train_reg, y_train_cls)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

for epoch in range(30):
    model.train()
    losses = []
    for xb, yb_reg, yb_cls in train_dl:
        xb, yb_reg, yb_cls = xb.to(device), yb_reg.to(device), yb_cls.to(device)
        pred_ret, pred_crash = model(xb)
        loss = mse_loss(pred_ret, yb_reg) + 0.5 * bce_loss(pred_crash, yb_cls)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}: Loss = {np.mean(losses):.6f}")

In [ ]:
# LSTM Forecast Plot
model.eval()
with torch.no_grad():
    pred_ret, pred_crash = model(X_test.to(device))
    pred_ret = pred_ret.cpu().numpy()
    pred_crash_prob = torch.sigmoid(pred_crash).cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

t = np.arange(len(y_test_reg))
axes[0].plot(t, y_test_reg, color='grey', lw=0.8, alpha=0.7, label='Actual')
axes[0].plot(t, pred_ret, color=MAIN_BLUE, lw=1.5, label='LSTM Forecast')
axes[0].axhline(0, color='grey', lw=0.5)
axes[0].set_title('LSTM Return Forecast vs Actual', fontweight='bold')
axes[0].set_ylabel('Log Return'); axes[0].legend()

axes[1].plot(t, pred_crash_prob, color=IDA_RED, lw=1.5, label='Crash Probability')
axes[1].fill_between(t, pred_crash_prob, alpha=0.2, color=IDA_RED)
actual_crashes = np.where(y_test_cls == 1)[0]
if len(actual_crashes) > 0:
    axes[1].scatter(actual_crashes, [1.0]*len(actual_crashes), color=CRIMSON,
                    marker='v', s=40, zorder=5, label='Actual Crashes')
axes[1].axhline(0.5, color='grey', ls='--', lw=1)
axes[1].set_title('Crash Probability Forecast', fontweight='bold')
axes[1].set_ylabel('P(crash)'); axes[1].legend()
axes[1].set_xlabel('Time')

plt.tight_layout()
save_fig('nn_ch05_crypto_lstm_forecast')

In [ ]:
# Crash Prediction ROC
from sklearn.metrics import roc_curve, auc

if y_test_cls.sum() > 0:
    fpr, tpr, _ = roc_curve(y_test_cls, pred_crash_prob)
    roc_auc = auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(fpr, tpr, color=MAIN_BLUE, lw=2.5, label=f'LSTM (AUC = {roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], color='grey', ls='--', lw=1)
    ax.fill_between(fpr, tpr, alpha=0.15, color=MAIN_BLUE)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC - Crypto Crash Prediction', fontweight='bold')
    ax.legend(fontsize=12)
    save_fig('nn_ch05_crypto_roc')
else:
    print("No crashes in test set; creating placeholder")
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.text(0.5, 0.5, 'No crash events in test period', ha='center', va='center', fontsize=14)
    ax.set_title('ROC - Crypto Crash Prediction', fontweight='bold')
    save_fig('nn_ch05_crypto_roc')

## Summary

Trained a multi-task **LSTM** for Bitcoin return forecasting and crash prediction. The model outputs both a return forecast and crash probability, evaluated via time-series plots and ROC analysis.